Build an image classification model using a pretrained CNN and apply transfer learning to CIFAR-10. Compare feature extraction and fine-tuning with a CNN trained from scratch to understand how pretrained models can improve performance and reduce training effort.

For Project B, we'll use CIFAR-10 again initially so that you can directly compare this project with your Project A.

In [1]:
#Import TensorFlow and CIFAR-10
import tensorflow as tf
from tensorflow.keras.datasets import cifar10

(X_train,y_train),(X_test,y_test)=cifar10.load_data()

In [2]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(50000, 32, 32, 3)
(50000, 1)
(10000, 32, 32, 3)
(10000, 1)


Our CIFAR-10 images are:
32 × 32 × 3
We'll resize them because the pretrained model we're going to use expects a larger input.

In [7]:
#Resize the images

X_train=tf.image.resize(X_train,(96,96))
X_test=tf.image.resize(X_test,(96,96))




Now:
CIFAR-10 image
32 × 32 × 3
      ↓
96 × 96 × 3

In [3]:
#Load MobileNetV2 with pretrained weights

from tensorflow.keras.applications import MobileNetV2

base_model=MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96,96,3)

)
#9406464 is the size of the downloaded weights file in bytes.

input_shape=(96, 96, 3)
This tells MobileNetV2:
Our input images will be 96 × 96 pixels with 3 color channels (RGB).

include_top=False
The original MobileNetV2 has:

CNN layers
   ↓
Original ImageNet classifier
   ↓
1000 classes
We don't want the original 1000-class ImageNet classifier, because our task is CIFAR-10 with 10 classes.

So:
include_top=False
means:
Remove the original classification layers and keep the pretrained CNN feature-extraction part.

include_top=False means remove the original “top” classification part of MobileNetV2 — the part that produces the 1000 ImageNet classes.
It removes the original final classification part that gives the 1000-class prediction.

Why are we importing MobileNetV2 if it's an architecture?
MobileNetV2 is an architecture, but when we write:

MobileNetV2(weights="imagenet", ...)
we are asking TensorFlow to give us:
the MobileNetV2 architecture + the pretrained ImageNet weights

So we are not using an empty MobileNetV2 architecture.
We are using:

MobileNetV2 architecture
        +
ImageNet-learned weights
        ↓
PRETRAINED MobileNetV2
That's the model we're using for transfer learning

ImageNet is a dataset. ✅
We are not saying ImageNet itself contains learned weights.

The process is:
ImageNet dataset
      ↓
Train MobileNetV2 architecture
      ↓
Model learns weights
      ↓
Save those learned weights
      ↓
Pretrained MobileNetV2

So when we write:
MobileNetV2(weights="imagenet")
"imagenet" means:

Load the MobileNetV2 weights that were previously learned by training on the ImageNet dataset.

So:
ImageNet = dataset
ImageNet-trained weights = learned weights produced from that dataset ✅

MobileNetV2 architecture
        +
weights learned by training on ImageNet
        ↓
Pretrained MobileNetV2 model
So:
MobileNetV2(weights="imagenet")
means “give me the MobileNetV2 architecture with its previously learned ImageNet weights.”

why given input shape?

Because the model needs to know what shape of images it will receive as input.

input_shape=(96, 96, 3)

means:

96 → height
96 → width
3 → RGB channels

Our CIFAR-10 images were originally:

32 × 32 × 3

We resized them to:

96 × 96 × 3

So we tell MobileNetV2:

“The images I am going to give you are 96×96 RGB images.” ✅

In [4]:
base_model.summary()

Model: "mobilenetv2_1.00_96"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 96, 96, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 48, 48,    │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 48, 48,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 48, 48,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 48, 48,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 48, 48,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 48, 48,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 48, 48,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 48, 48,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 48, 48,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 48, 48,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 48, 48,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 49, 49,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 24, 24,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 24, 24,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 24, 24,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 24, 24,    │      2,304 │ block_1_depthwis

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 2,223,872 (8.48 MB)

 Non-trainable params: 34,112 (133.25 KB)

Input
(None, 96, 96, 3)
        ↓
many CNN layers / blocks
        ↓
feature representation

And because we used:

include_top=False
the original 1000-class classification head is not present.

In [5]:
base_model.trainable=False

base_model.trainable = False

means:This will freeze the pretrained MobileNetV2 weights so that, initially, we train only our new classifier.
Use the pretrained MobileNetV2 with its already-learned weights, but do not update those weights during our training.

We are taking the pretrained MobileNetV2 and adding a small new classifier on top of it.
Think of it like:

Pretrained MobileNetV2
(already learned visual features)
        ↓
   Feature extractor
        ↓
 NEW classifier
        ↓
  10 CIFAR-10 classes

The MobileNetV2 part is already the CNN.
The new classifier is mainly responsible for taking the features produced by MobileNetV2 and deciding:
“Which of our 10 classes does this image belong to?”

So:
New CNN from scratch ❌
Pretrained CNN + new classification head ✅
And in this first stage, the pretrained CNN is frozen, so only the new classifier learns.

We use the pretrained MobileNetV2’s already-learned weights, freeze that part so its weights do not change during training, and then train only our new classifier, whose weights are updated to learn how to map MobileNetV2’s extracted features to our 10 CIFAR-10 classes.

But, why training that new classifier again?

Because the pretrained classifier was trained for ImageNet's 1000 classes, not our CIFAR-10's 10 classes.
So we remove that old classifier and create a new one that learns:

MobileNetV2 features
        ↓
NEW classifier
        ↓
10 CIFAR-10 classes

The MobileNetV2 already knows how to extract useful visual features; the new classifier needs to learn how to use those features for our specific classes.
That's why we train the new classifier.

 We remove the pretrained model's original classification head (because it predicts 1000 ImageNet classes), then add our own new classifier for the 10 CIFAR-10 classes.

Pretrained MobileNetV2
        ↓
remove original classifier ❌
        ↓
pretrained feature extractor ✅
        ↓
add our classifier ✅
        ↓
10 CIFAR-10 classes

Are we doing a type of fine-tuning here? Tell shortly. beause we are adding our features

No. ❌

This is feature extraction, not fine-tuning.

We are adding a new classifier, but the pretrained MobileNetV2 layers remain frozen.

Fine-tuning = unfreeze some pretrained layers and train them further.

If the pretrained model had some final classification layer whose classes didn't match our task, we would replace that classification layer with our new classifier.

But if you remove a feature-extraction layer in the CNN (a Conv/BatchNorm/etc. layer), you don't automatically need another classifier.

So the key is:

Wrong final classifier → replace it. ✅
Remove a feature layer → no new classifier just because of that. ✅

In [6]:
#Now we build our new classifier on top of the frozen MobileNetV2.

from tensorflow.keras import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D,Dense

model=Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128,activation='relu'),
    Dense(10,activation='softmax')
])

Here:

base_model → pretrained MobileNetV2, whose weights are frozen.
GlobalAveragePooling2D → converts the feature maps into a smaller vector.
Dense(128) → learns from those features.
Dense(10) → our new classifier for the 10 CIFAR-10 classes.

Yes. ✅ The new classifier is the new final layer(s) we add after the pretrained MobileNetV2 to classify our 10 CIFAR-10 classes 


Think of classifier = the new part we add after MobileNetV2:

MobileNetV2 (pretrained)
        ↓
    Classifier
        ↓
   10 classes

In our code, we made that classifier using two Dense layers:

Dense(128, activation="relu")
Dense(10, activation="softmax")

Why two? The first layer processes the features, and the second layer finally chooses among the 10 classes.

In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

MobileNetV2
(None, 3, 3, 1280)
This is the pretrained CNN.

It has 2,257,984 parameters, which are the weights it learned from ImageNet.
For now, we will keep these frozen.


GlobalAveragePooling2D
(None, 1280)

MobileNetV2 gives us a feature map of:
3 × 3 × 1280    So there are 1280 feature maps, each of size 3×3.

This layer converts that into:
1280

So it basically turns the CNN's feature maps into a 1280-number feature vector that our classifier can work with.


MobileNetV2 produces features → GlobalAveragePooling converts those features into 1280 numbers → our classifier uses those numbers.

GlobalAveragePooling2D is NOT converting 1000 classes. Because we used include_top=False, the 1000-class classifier was already removed.

Instead:

Pretrained MobileNetV2
        ↓
3 × 3 × 1280 feature maps
        ↓
GlobalAveragePooling2D
        ↓
1280-number feature vector
        ↓
Dense(128)
        ↓
Dense(10) → final 10-class output

So yes, we have our base model first, then a pooling layer, then our new Dense layer, and finally our 10-class output layer. The pooling layer simply converts the MobileNetV2 feature maps into a compact vector that our new classifier can use.

In [8]:
#right now we've built the model, but we haven't told TensorFlow how to train it yet. 

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

look at trainable vs non-trainable parameters.

That is important because we want to verify:

MobileNetV2 weights → NOT updating 🔒
Our new classifier → updating ✅

Total parameters = all learnable numbers/weights in the entire model:
2,423,242 = pretrained MobileNetV2 + our new classifier.

Trainable parameters = the parameters that are allowed to change during training:
165,258 = our new classifier.  165,258 new classifier weights → trainable ✅

Non-trainable parameters = the pretrained MobileNetV2 parameters that we froze:
2,257,984.

And yes, feature extraction means we're using the features learned by the pretrained MobileNetV2 and passing those features to our new classifier. We are not changing the pretrained features in this stage; we're using them as they are.
One tiny correction: we're not only "extracting features" from the final layer. The whole frozen MobileNetV2 feature-extraction part processes the image and produces useful features, which our new classifier then uses.

In [10]:
#Now we train it.
#Before that, one important preprocessing step: MobileNetV2 expects its own input scaling, so we'll use its preprocessing function rather than simply dividing by 255.
#preprocess_input → matches MobileNetV2's expected input format.

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
X_train=preprocess_input(X_train)
X_test=preprocess_input(X_test)



The pretrained model knows the scaling it was trained with, but it does not automatically rescale new images for us.

We are giving it CIFAR-10 images, not the original ImageNet images.

So we must convert our CIFAR-10 pixel values into the same input format MobileNetV2 expects.

ImageNet training:
images → required preprocessing → MobileNetV2

Our project:
CIFAR-10 images → same preprocessing → MobileNetV2

So preprocess_input() is basically saying:

“Prepare our new images in the same format the pretrained model was trained to receive.” ✅